# Delivery times of three apps: mean ± SD, sorted

**Box Plot** · a worked example from the StatNex teaching guide

**The story.** Management wants one slide comparing the delivery times of three food-delivery apps, ZipEats, FeastNow and UrbanBite, including any unusual orders. The table holds the delivery time in minutes of 50 orders from each app.

**The question.** Which app is fastest, which is most consistent, and which orders are unusual?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/box_plot.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,ZipEats,FeastNow,UrbanBite
1.0,16.0,32.0,36.0
2.0,24.0,26.0,41.0
3.0,28.0,31.0,38.0
4.0,19.0,33.0,48.0
5.0,29.0,25.0,33.0
6.0,26.0,37.0,42.0
7.0,22.0,30.0,42.0
8.0,30.0,28.0,42.0
9.0,22.0,29.0,34.0
10.0,26.0,35.0,46.0
11.0,28.0,33.0,38.0
12.0,17.0,27.0,53.0
13.0,29.0,34.0,41.0
14.0,26.0,26.0,45.0
15.0,23.0,23.0,35.0
16.0,23.0,30.0,46.0
17.0,23.0,30.0,47.0
18.0,22.0,33.0,33.0
19.0,26.0,35.0,44.0
20.0,12.0,29.0,31.0
21.0,26.0,29.0,32.0
22.0,14.0,42.0,44.0
23.0,29.0,33.0,42.0
24.0,20.0,32.0,39.0
25.0,23.0,32.0,41.0
26.0,20.0,41.0,38.0
27.0,27.0,29.0,46.0
28.0,23.0,36.0,38.0
29.0,28.0,34.0,36.0
30.0,19.0,33.0,31.0
31.0,27.0,30.0,36.0
32.0,27.0,32.0,38.0
33.0,29.0,38.0,32.0
34.0,24.0,35.0,46.0
35.0,26.0,41.0,45.0
36.0,18.0,31.0,38.0
37.0,25.0,37.0,44.0
38.0,58.0,37.0,34.0
39.0,17.0,33.0,32.0
40.0,31.0,31.0,29.0
41.0,27.0,40.0,45.0
42.0,18.0,39.0,40.0
43.0,24.0,35.0,37.0
44.0,29.0,37.0,42.0
45.0,25.0,35.0,43.0
46.0,24.0,35.0,36.0
47.0,25.0,31.0,39.0
48.0,23.0,33.0,40.0
49.0,20.0,37.0,46.0
50.0,26.0,33.0,44.0

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per variable. Blank and text cells are left out, so columns may differ
# in length.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["ZipEats", "FeastNow", "UrbanBite"]
series = {c: pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float) for c in columns}

## Descriptive statistics

In [ ]:
# Variance and SD divide by n - 1. Quartiles interpolate linearly between the sorted
# values (NumPy's default, Excel's QUARTILE.INC).
def describe(x):
    q1, median, q3 = np.percentile(x, [25, 50, 75])
    return {"n": x.size, "mean": np.mean(x), "variance": np.var(x, ddof=1),
            "sd": np.std(x, ddof=1), "min": x.min(), "q1": q1, "median": median,
            "q3": q3, "max": x.max(), "iqr": q3 - q1}


summary = pd.DataFrame({name: describe(x) for name, x in series.items()})

## Box plot data

In [ ]:
# Five points per variable, top to bottom: upper whisker, box top, centre line, box
# bottom, lower whisker. What each one is depends on how the plot is drawn.
# The box is the mean ± one standard deviation (n - 1) and the line in it the mean;
# the whiskers run to the smallest and the largest value, so no value is an outlier.
labels = ["maximum", "mean + SD", "mean", "mean - SD", "minimum"]


def box(x):
    mean, sd = np.mean(x), np.std(x, ddof=1)
    return {"five": [x.max(), mean + sd, mean, mean - sd, x.min()],
            "upper outliers": [], "lower outliers": []}


boxes = {name: box(x) for name, x in series.items()}
# Variables in descending order of the centre line (the median; the mean when the box
# is mean ± SD). Ties keep the order the columns were chosen in.
order = sorted(boxes, key=lambda name: boxes[name]["five"][2], reverse=True)
summary = summary[order]
boxes = {name: boxes[name] for name in order}
print(summary)

# Upper outliers above the whisker (largest first), lower ones below it (smallest
# first); a variable with fewer outliers than another leaves the cell empty.
most_upper = max(len(b["upper outliers"]) for b in boxes.values())
most_lower = max(len(b["lower outliers"]) for b in boxes.values())


def column(b):
    upper = b["upper outliers"] + [np.nan] * (most_upper - len(b["upper outliers"]))
    lower = b["lower outliers"] + [np.nan] * (most_lower - len(b["lower outliers"]))
    return upper + b["five"] + lower


boxdata = pd.DataFrame({name: column(b) for name, b in boxes.items()},
                       index=["upper outlier"] * most_upper + labels
                       + ["lower outlier"] * most_lower)
print()
print(boxdata)

## Box plot

In [ ]:
# Drawn from the numbers above, so the picture and the table cannot disagree.
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots()
ax.bxp([{"label": name, "whishi": b["five"][0], "q3": b["five"][1], "med": b["five"][2],
         "q1": b["five"][3], "whislo": b["five"][4],
         "fliers": b["lower outliers"] + b["upper outliers"]} for name, b in boxes.items()])
fig.savefig("figures/box_plot.png", dpi=150)
plt.close(fig)

## What the result says

This box shows the mean (the average) and one standard deviation (SD) on each side of it. The SD is the typical distance of an order from the mean. The apps are sorted from the highest mean to the lowest: UrbanBite 39.76 minutes, FeastNow 32.94, ZipEats 24.46.

ZipEats is still fastest. But its SD is now the largest, 6.49 minutes against 4.23 for FeastNow and 5.34 for UrbanBite, so its box is the longest, from about 18 to 31 minutes. One order of 58 minutes causes this. The SD reacts strongly to one extreme value.

So this picture makes ZipEats look the least consistent, although its interquartile range, the spread of the middle half of its orders, is only 5 minutes.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.